In [9]:
from google.colab import drive

# Attempt to unmount the drive if it's already mounted
try:
    drive.flush_and_unmount()
    print('Drive unmounted successfully.')
except Exception as e:
    print(f'Drive was not mounted or could not be unmounted: {e}')

Drive unmounted successfully.


In [42]:
import os, glob, pickle, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import Point, box
from scipy.ndimage import label as scipy_label
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATASET_DIR         = '/content/drive/MyDrive/CS682/project/RiverScope_dataset'
RESULTS_DIR         = '/content/drive/MyDrive/CS682/project/results'
WIDTH_OUT_DIR       = os.path.join(RESULTS_DIR, 'width_estimation_new_run')
CACHE_PATH          = os.path.join(WIDTH_OUT_DIR, 'perp_cache_v2.pkl')
os.makedirs(WIDTH_OUT_DIR, exist_ok=True)

PLANET_RES  = 3      # meters per pixel
STEP        = 0.5    # walk step along perpendicular (matches RiverScope script)
DELTA       = 1      # slope computation delta
N_SLOPE     = 10     # neighbors for slope computation
MAX_DIST    = 200    # max perpendicular search distance in steps

test_df     = pd.read_csv(os.path.join(DATASET_DIR, 'test.csv'))
sword_nodes = pd.read_csv(os.path.join(DATASET_DIR, 'SWORD/SWORD_nodes.csv'))
sword_nodes['reach_id'] = sword_nodes['reach_id'].astype(int)

EXPERIMENTS = {
    'LP_224':   os.path.join(RESULTS_DIR, 'olmoearth_zero_shot_linear_probe_paper_approach/predictions'),
    'LP_448':   os.path.join(RESULTS_DIR, 'olmoearth_lp_448/predictions'),
    'CNN_448':  os.path.join(RESULTS_DIR, 'olmoearth_zero_shot_cnn_448/predictions'),
    'UNet_448': os.path.join(RESULTS_DIR, 'olmoearth_unet_448_new/predictions'),
    # 'UNet_496_7e5_boundary_loss': os.path.join(RESULTS_DIR, 'olmoearth_unet_496_new/predictions'),
    'UNet_496_5e4_old': os.path.join(RESULTS_DIR, 'olmoearth_unet_496_new/predictions_496_5e4'),
    'UNet_496': os.path.join(RESULTS_DIR, 'olmoearth_unet_496_v2_LATEST/predictions'),
}
print('Setup complete.')

Mounted at /content/drive
Setup complete.


In [35]:
EXPERIMENTS

{'LP_224': '/content/drive/MyDrive/CS682/project/results/olmoearth_zero_shot_linear_probe_paper_approach/predictions',
 'LP_448': '/content/drive/MyDrive/CS682/project/results/olmoearth_lp_448/predictions',
 'CNN_448': '/content/drive/MyDrive/CS682/project/results/olmoearth_zero_shot_cnn_448/predictions',
 'UNet_448': '/content/drive/MyDrive/CS682/project/results/olmoearth_unet_448_new/predictions',
 'UNet_496': '/content/drive/MyDrive/CS682/project/results/olmoearth_unet_496_v2_LATEST/predictions'}

In [36]:
print('Experiment pred_dirs:')
for name, path in EXPERIMENTS.items():
    exists = os.path.exists(path)
    # Escape the path because it contains brackets like [LATEST]
    search_pattern = os.path.join(glob.escape(path), '*.tif')
    n_tifs = len(glob.glob(search_pattern)) if exists else 0
    status = f'{n_tifs} .tifs' if exists else 'NOT FOUND'
    print(f'  {name:20s}: {status}')

Experiment pred_dirs:
  LP_224              : 235 .tifs
  LP_448              : 235 .tifs
  CNN_448             : 235 .tifs
  UNet_448            : 235 .tifs
  UNet_496            : 235 .tifs


In [18]:
def get_local_slope(line, pt, delta=1, N=10):
    """Compute slope at point along reach line using N neighbors."""
    distances = [line.project(pt) + delta * (i - N // 2) for i in range(N)]
    coords    = [line.interpolate(d).coords[0] for d in distances]
    x_vals, y_vals = zip(*coords)
    dx  = np.gradient(x_vals)
    dy  = np.gradient(y_vals)
    return np.arctan2(np.mean(dy), np.mean(dx))   # radians


def remove_small_islands(binary_mask, min_size=500):
    """Remove connected components smaller than min_size pixels."""
    labeled, n = scipy_label(binary_mask)
    for i in range(1, n + 1):
        if np.sum(labeled == i) < min_size:
            binary_mask[labeled == i] = 0
    return binary_mask


def get_sword_intersecting_mask(raster_path, sword_gdf):
    """Keep only water regions that intersect the SWORD reach — matches RiverScope."""
    from rasterio.features import shapes
    from shapely.geometry import shape
    from scipy import ndimage

    with rasterio.open(raster_path) as src:
        binary_mask = src.read(1).astype(np.uint8)
        transform   = src.transform
        raster_crs  = src.crs

    if sword_gdf.crs != raster_crs:
        sword_gdf = sword_gdf.to_crs(raster_crs)

    mask_labels, _ = ndimage.label(binary_mask)
    shapes_list     = list(shapes(mask_labels, transform=transform))
    region_polygons = [shape(geom) for geom, val in shapes_list if val > 0]
    region_ids      = [val        for geom, val in shapes_list if val > 0]

    gdf_regions   = gpd.GeoDataFrame({'region_id': region_ids,
                                       'geometry': region_polygons}, crs=raster_crs)
    intersecting  = gdf_regions[gdf_regions.intersects(sword_gdf.union_all())]
    inter_ids     = intersecting['region_id'].tolist()
    inter_mask    = np.isin(mask_labels, inter_ids)

    return binary_mask, inter_mask.astype(np.uint8), transform, raster_crs


print('Helper functions defined.')

Helper functions defined.


In [19]:
gt_widths_file = os.path.join(DATASET_DIR, 'PlanetScope', 'derived_gt_widths-test.csv')
gt_widths_df = pd.read_csv(gt_widths_file)

print(f"✓ Loaded {len(gt_widths_df)} GT width measurements")
print(f"Columns: {gt_widths_df.columns.tolist()}")

✓ Loaded 541 GT width measurements
Columns: ['node_id', 'reach_id', 'width_m']


In [20]:
import pickle
with open(CACHE_PATH, 'rb') as f:
    cache = pickle.load(f)


In [22]:
# Check a sample filename from LP_224
import glob, os
# lp_224_files = glob.glob(os.path.join(EXPERIMENTS['LP_224'], '*.tif'))
if lp_224_files:
    print("LP_224 Sample Key:", os.path.splitext(os.path.basename(lp_224_files[0]))[0])

# Check a sample key existing in cache
sample_cache_key = list(cache.keys())[0] if cache else "Cache is empty"
print("Cache Sample Key :", sample_cache_key)


LP_224 Sample Key: 20230615_134909_40_2424_3B_AnalyticMS_SR_clip-tile_1202_1702-967-1467_mid
Cache Sample Key : 20230601_023106_78_24af_3B_AnalyticMS_SR_clip-tile_454_954-1224-1724_mid


In [37]:
# --- OPTIMIZATION 1: Index test_df for instant lookups ---
test_lookup = test_df.set_index('normalized_planetscope_path').to_dict('index')

# --- OPTIMIZATION 2: Convert all nodes to geometries ONCE ---
print("Pre-processing sword_nodes to GeoDataFrame...")
sword_gdf_all = gpd.GeoDataFrame(
    sword_nodes,
    geometry=gpd.points_from_xy(sword_nodes['x'], sword_nodes['y']),
    crs='EPSG:4326'
)

# --- OPTIMIZATION 3: Pre-group by reach_id for instant O(1) access ---
print("Indexing nodes by reach_id...")
nodes_by_reach = {
    rid: group for rid, group in sword_gdf_all.groupby('reach_id')
}

REF_PRED_DIR = EXPERIMENTS['UNet_496']
# REF_PRED_DIR = os.path.join(glob.escape(REF_PRED_DIR), '*.tif')
pred_files   = sorted(glob.glob(os.path.join(REF_PRED_DIR, '*.tif')))
shp_cache    = {}
cache        = {}

for pred_path in tqdm(pred_files, desc='Precomputing perpendiculars'):
    tile_name = os.path.splitext(os.path.basename(pred_path))[0]
    norm_path = os.path.join('PlanetScope', 'input', 'test', os.path.basename(pred_path))

    # Lookup 1: Instant dictionary access instead of scanning dataframe
    row = test_lookup.get(norm_path)
    if not row:
        continue

    reach_id = int(row['reach_id'])
    shp_path = os.path.join(DATASET_DIR, row['sword_path'])

    # Lookup 2: Instant dictionary access instead of scanning sword_nodes
    reach_nodes = nodes_by_reach.get(reach_id)
    if reach_nodes is None or len(reach_nodes) == 0:
        continue

    # Get tile CRS and bounds
    try:
        with rasterio.open(pred_path) as src:
            raster_crs    = src.crs
            raster_bounds = src.bounds
            transform     = src.transform
            H, W          = src.height, src.width
    except Exception:
        continue

    # Convert CRS bounds to 4326 for fast overlap filtering
    bounds_4326 = transform_bounds(
        raster_crs, 'EPSG:4326',
        raster_bounds.left, raster_bounds.bottom,
        raster_bounds.right, raster_bounds.top
    )
    lon_min, lat_min, lon_max, lat_max = bounds_4326

    # Fast Vectorized Geopandas filter
    tile_nodes = reach_nodes[
        (reach_nodes['x'] >= lon_min) & (reach_nodes['x'] <= lon_max) &
        (reach_nodes['y'] >= lat_min) & (reach_nodes['y'] <= lat_max)
    ]

    if len(tile_nodes) == 0:
        continue

    # Load shapefile using existing cache
    if shp_path not in shp_cache:
        try:
            shp_cache[shp_path] = gpd.read_file(shp_path)
        except:
            continue

    sword_gdf = shp_cache[shp_path]
    reach_gdf = sword_gdf[sword_gdf['reach_id'] == reach_id].to_crs(raster_crs)
    if len(reach_gdf) == 0:
        continue

    try:
        reach_line = reach_gdf.geometry.union_all()
    except Exception:
        continue

    # Reproject only the tiny subset of nodes for this tile
    nodes_gdf_local = tile_nodes.to_crs(raster_crs)

    tile_cache = []
    for _, node_row in nodes_gdf_local.iterrows():
        gt_width = float(node_row.get('width', np.nan))
        # Matches original constraint
        if np.isnan(gt_width) or gt_width > 500:
            continue

        pt    = node_row['geometry']
        pt_x, pt_y  = pt.x, pt.y

        # Check node inside bounds
        try:
            r_check, c_check = rasterio.transform.rowcol(transform, pt_x, pt_y)
            if not (0 <= r_check < H and 0 <= c_check < W):
                continue
        except Exception:
            continue

        # Compute perpendicular
        try:
            slope      = get_local_slope(reach_line, pt, delta=DELTA, N=N_SLOPE)
            perp_angle = slope + np.pi / 2
            dx, dy     = np.cos(perp_angle), np.sin(perp_angle)
        except Exception:
            continue

        tile_cache.append({
            'node_id':  int(node_row['node_id']),
            'gt_width': gt_width,
            'pt_x':     pt_x, 'pt_y': pt_y,
            'dx':       dx,   'dy':   dy,
        })

    if tile_cache:
        cache[tile_name] = {
            'nodes':     tile_cache,
            'shp_path':  shp_path,
            'reach_id':  reach_id,
            'norm_path': norm_path,
        }

# Save final computed cache
with open(CACHE_PATH, 'wb') as f:
    pickle.dump(cache, f)

print(f'\nOptimization Complete!')
print(f'Tiles: {len(cache)}  |  Nodes: {sum(len(v["nodes"]) for v in cache.values())}')


Pre-processing sword_nodes to GeoDataFrame...
Indexing nodes by reach_id...


Precomputing perpendiculars:   0%|          | 0/235 [00:00<?, ?it/s]


Optimization Complete!
Tiles: 212  |  Nodes: 1586


In [38]:
import pandas as pd
import os

# CORRECTED: GT widths are in PlanetScope, not SWORD
gt_widths_file = os.path.join(DATASET_DIR, 'PlanetScope', 'derived_gt_widths-test.csv')

print(f"Loading GT widths from: {gt_widths_file}")
gt_widths_df = pd.read_csv(gt_widths_file)

print(f"✓ Loaded {len(gt_widths_df)} GT width measurements")
print(f"\nColumns: {gt_widths_df.columns.tolist()}")
print(f"\nFirst 10 rows:")
print(gt_widths_df.head(10))
print(f"\nGT Width Statistics:")
print(gt_widths_df['width_m'].describe())
print(f"\nUnique node_ids: {gt_widths_df['node_id'].nunique()}")

Loading GT widths from: /content/drive/MyDrive/CS682/project/RiverScope_dataset/PlanetScope/derived_gt_widths-test.csv
✓ Loaded 541 GT width measurements

Columns: ['node_id', 'reach_id', 'width_m']

First 10 rows:
          node_id     reach_id  width_m
0  22733000090313  22733000093    228.0
1  22733000090323  22733000093    792.0
2  22733000090333  22733000093   1302.0
3  22733000090343  22733000093   1896.0
4  22733000090353  22733000093   2217.0
5  22733000090363  22733000093   3672.0
6  22733000090373  22733000093   3600.0
7  22733000090383  22733000093   4200.0
8  22733000090393  22733000093   4203.0
9  22733000090403  22733000093   3600.0

GT Width Statistics:
count     541.000000
mean      431.894862
std       755.704586
min         3.000000
25%       100.500000
50%       178.200000
75%       326.571429
max      4728.000000
Name: width_m, dtype: float64

Unique node_ids: 541


In [ ]:
# ls -la /content/drive/MyDrive/CS682/project/RiverScope_dataset/PlanetScope/

total 33
-rw------- 1 root root 18251 May 11  2025 derived_gt_widths-test.csv
-rw------- 1 root root  6148 Apr  1 00:03 .DS_Store
drwx------ 2 root root  4096 Apr 20 00:32 input/
drwx------ 2 root root  4096 Apr 20 00:32 label/


In [ ]:
# # See what's in it:
# !head -5 /content/drive/MyDrive/RiverScope/RiverScope_dataset/SWORD/derived_gt_widths-test.csv

# # Count how many GT widths:
# !wc -l /content/drive/MyDrive/RiverScope/RiverScope_dataset/SWORD/derived_gt_widths-test.csv

head: cannot open '/content/drive/MyDrive/RiverScope/RiverScope_dataset/SWORD/derived_gt_widths-test.csv' for reading: No such file or directory
wc: /content/drive/MyDrive/RiverScope/RiverScope_dataset/SWORD/derived_gt_widths-test.csv: No such file or directory


In [39]:
def estimate_widths_for_experiment_ROBUST(pred_dir, cache, gt_widths_df, step=STEP, resolution=PLANET_RES):
    gt_width_lookup = dict(zip(gt_widths_df['node_id'], gt_widths_df['width_m']))

    # --- NEW: Pre-index cache by base filename ---
    # Maps '20230601...SR_clip' -> [List of all cached tile data objects]
    cache_by_base = {}
    for key, data in cache.items():
        base_key = key.split('-tile')[0]
        if base_key not in cache_by_base:
            cache_by_base[base_key] = []
        cache_by_base[base_key].append(data)

    results = []
    pred_files = sorted(glob.glob(os.path.join(pred_dir, '*.tif')))

    for pred_path in tqdm(pred_files, desc='Width estimation', leave=False):
        full_filename = os.path.splitext(os.path.basename(pred_path))[0]

        # --- REVISED LOOKUP LOGIC ---
        matched_tile_infos = []

        # 1. Try exact match first (e.g. UNet tiled outputs)
        if full_filename in cache:
            matched_tile_infos = [cache[full_filename]]
        else:
            # 2. Try fallback to base match (e.g. LP_224 full scene outputs)
            base_key = full_filename.split('-tile')[0]
            matched_tile_infos = cache_by_base.get(base_key, [])

        if not matched_tile_infos:
            continue  # No overlap found in cache for this file

        # Load predicted water mask (Read once for the whole file)
        try:
            with rasterio.open(pred_path) as src:
                pred_mask = src.read(1).astype(np.uint8)
                transform = src.transform
                H, W = src.shape
        except Exception:
            continue

        # Iterate over all collected tile segments linked to this image
        for tile_info in matched_tile_infos:
            nodes    = tile_info['nodes']
            reach_id = tile_info.get('reach_id')

            # Process each SWORD node
            for node_info in nodes:
                node_id, pt_x, pt_y, dx, dy = node_info['node_id'], node_info['pt_x'], node_info['pt_y'], node_info['dx'], node_info['dy']

                gt_width = gt_width_lookup.get(node_id)
                if gt_width is None:
                    continue
                gt_width = float(gt_width)

                # Pixel walking logic
                max_dist = 200 * step
                closest_water = None
                min_dist = float('inf')
                for direction in [-1, 1]:
                    for i in range(1, int(max_dist / step)):
                        new_x = pt_x + direction * dx * step * i
                        new_y = pt_y + direction * dy * step * i
                        try:
                            r, c = rasterio.transform.rowcol(transform, new_x, new_y)
                            if 0 <= r < H and 0 <= c < W and pred_mask[r, c] == 1:
                                dist = np.hypot(new_x - pt_x, new_y - pt_y)
                                if dist < min_dist:
                                    min_dist = dist
                                    closest_water = (new_x, new_y)
                        except:
                            continue
                if closest_water is None:
                    continue

                all_row_col = []
                row_pt, col_pt = rasterio.transform.rowcol(transform, pt_x, pt_y)
                cw_x, cw_y = closest_water
                try:
                    if 0 <= row_pt < H and 0 <= col_pt < W and pred_mask[row_pt, col_pt] == 1:
                        directions, start_x, start_y = [(-1, 'l'), (1, 'r')], pt_x, pt_y
                    else:
                        direction = np.sign((cw_x - pt_x) * dx + (cw_y - pt_y) * dy)
                        if direction == 0: continue
                        directions, start_x, start_y = [(direction, 'x')], cw_x, cw_y
                except:
                    continue

                for sign, _ in directions:
                    i = 1
                    while True:
                        nx, ny = start_x + sign * dx * step * i, start_y + sign * dy * step * i
                        try:
                            r, c = rasterio.transform.rowcol(transform, nx, ny)
                            if 0 <= r < H and 0 <= c < W and pred_mask[r, c] == 1:
                                all_row_col.append((r, c))
                            else:
                                break
                        except:
                            break
                        i += 1

                water_px = len(set(all_row_col))
                if water_px > 0:
                    results.append({
                        'node_id': node_id,
                        'reach_id': reach_id,
                        'gt_width': gt_width,
                        'pred_width': water_px * resolution,
                        'num_water_px': water_px,
                    })

    return pd.DataFrame(results)


In [40]:
import pickle

# Load the precomputed cache from disk
print(f"Loading cache from {CACHE_PATH}...")
with open(CACHE_PATH, 'rb') as f:
    cache = pickle.load(f)

print(f"Successfully loaded cache with {len(cache)} entries.")


Loading cache from /content/drive/MyDrive/CS682/project/results/width_estimation_new_run/perp_cache_v2.pkl...
Successfully loaded cache with 212 entries.


In [ ]:
# # Check a sample filename from LP_224
# import glob, os
# lp_224_files = glob.glob(os.path.join(EXPERIMENTS['LP_224'], '*.tif'))
# if lp_224_files:
#     print("LP_224 Sample Key:", os.path.splitext(os.path.basename(lp_224_files[0]))[0])

# # Check a sample key existing in cache
# sample_cache_key = list(cache.keys())[0] if cache else "Cache is empty"
# print("Cache Sample Key :", sample_cache_key)


LP_224 Sample Key: 20230615_134909_40_2424_3B_AnalyticMS_SR_clip
Cache Sample Key : 20230601_023106_78_24af_3B_AnalyticMS_SR_clip-tile_454_954-1224-1724_mid


In [43]:
import os
import pickle

ALL_DFS_PATH = os.path.join(WIDTH_OUT_DIR, 'all_experiment_dfs.pkl')

# 1. Load from cache if exists
if os.path.exists(ALL_DFS_PATH):
    print(f"Checking cache at {ALL_DFS_PATH}...")
    with open(ALL_DFS_PATH, 'rb') as f:
        all_dfs = pickle.load(f)
else:
    print("No cache found. Initializing empty storage.")
    all_dfs = {}

# 2. SMART CACHE CHECK: Check each experiment in your config
needs_update = False

for name, pred_dir in EXPERIMENTS.items():
    # Check if experiment wasn't calculated OR has 0 rows (like previous LP_224)
    if name not in all_dfs or all_dfs[name].empty:
        print(f"--> Force recalculating missing or empty model: {name}")
        all_dfs[name] = estimate_widths_for_experiment_ROBUST(
            pred_dir,
            cache,
            gt_widths_df
        )
        needs_update = True

# 3. ENSURE CACHE IS SAVED: If we recalculated anything, write it back to disk immediately
if needs_update:
    print(f"Saving updated experiment results to {ALL_DFS_PATH}...")
    with open(ALL_DFS_PATH, 'wb') as f:
        pickle.dump(all_dfs, f)
    print("Cache updated successfully!")
else:
    print("All experiments verified. Loaded everything from cache.")

# ======================================================
# METRICS LOGIC & PRINTING
# ======================================================
def compute_metrics(df):
    if df.empty:
        return {'n_nodes': 0, 'medae': float('nan'), 'mae': float('nan'), 'bias': float('nan')}
    errors = df['pred_width'] - df['gt_width']
    return {
        'n_nodes':  len(df),
        'medae':    float(errors.abs().median()),
        'mae':      float(errors.abs().mean()),
        'bias':     float(errors.mean()),
    }

print('═'*75)
print(f'{"Model":25s}  {"MedAE":>8}  {"MAE":>8}  {"Bias":>8}  {"n":>6}')
print('─'*75)

print('--- RiverScope Paper Baselines ---')
BASELINES = {
    'RiverScope (best)': {'medae': 7.2,  'mae': 15.3,  'bias':   5.7, 'n': 445},
    'PlanetScope CNN':   {'medae': 30.0, 'mae': 87.2,  'bias':  39.3, 'n': 445},
    'Sentinel':          {'medae': 39.0, 'mae': 152.8, 'bias': 119.8, 'n': 445},
    'SWOT':              {'medae': 41.9, 'mae': 94.4,  'bias':  28.2, 'n': 445},
    'SWORD/Landsat':     {'medae': 45.0, 'mae': 77.0,  'bias':  -6.9, 'n': 445},
}

for name, m in BASELINES.items():
    print(f'{name:25s}  {m["medae"]:>7.1f}m  {m["mae"]:>7.1f}m  '
          f'{m["bias"]:>+7.1f}m  {m["n"]:>6}')

print('\n--- Your Models ---')
for name, df in all_dfs.items():
    m = compute_metrics(df)
    print(f'{name:25s}  {m["medae"]:>7.1f}m  {m["mae"]:>7.1f}m  '
          f'{m["bias"]:>+7.1f}m  {m["n_nodes"]:>6}')


Checking cache at /content/drive/MyDrive/CS682/project/results/width_estimation_new_run/all_experiment_dfs.pkl...
--> Force recalculating missing or empty model: UNet_496_5e4_old


Width estimation:   0%|          | 0/235 [00:00<?, ?it/s]

Saving updated experiment results to /content/drive/MyDrive/CS682/project/results/width_estimation_new_run/all_experiment_dfs.pkl...
Cache updated successfully!
═══════════════════════════════════════════════════════════════════════════
Model                         MedAE       MAE      Bias       n
───────────────────────────────────────────────────────────────────────────
--- RiverScope Paper Baselines ---
RiverScope (best)              7.2m     15.3m     +5.7m     445
PlanetScope CNN               30.0m     87.2m    +39.3m     445
Sentinel                      39.0m    152.8m   +119.8m     445
SWOT                          41.9m     94.4m    +28.2m     445
SWORD/Landsat                 45.0m     77.0m     -6.9m     445

--- Your Models ---
LP_224                        15.0m     38.1m    -14.3m    1342
LP_448                        15.0m     38.9m    -15.5m    1366
CNN_448                       18.0m     38.1m     -0.8m    1369
UNet_448                      17.0m     39.0m     +2.0m

In [ ]:
import pickle

ALL_DFS_PATH = os.path.join(WIDTH_OUT_DIR, 'all_experiment_dfs.pkl')

with open(ALL_DFS_PATH, 'rb') as f:
    all_dfs = pickle.load(f)

print("Restored the following from disk:", list(all_dfs.keys()))


Restored the following from disk: ['LP_224', 'LP_448', 'CNN_448', 'UNet_448', 'UNet_496_7e5_boundary_loss', 'UNet_496_5e4_best']


In [ ]:
import pickle
import os

ALL_DFS_PATH = os.path.join(WIDTH_OUT_DIR, 'all_experiment_dfs.pkl')
with open(ALL_DFS_PATH, 'wb') as f:
    pickle.dump(all_dfs, f)

print(f"Saved to Google Drive!")


Saved to Google Drive!


In [ ]:
import os
from datetime import datetime

if os.path.exists(ALL_DFS_PATH):
    size_bytes = os.path.getsize(ALL_DFS_PATH)
    mod_time = datetime.fromtimestamp(os.path.getmtime(ALL_DFS_PATH))

    print("✅ Cache found on Drive!")
    print(f"Path: {ALL_DFS_PATH}")
    print(f"Size: {size_bytes / 1e6:.2f} MB")
    print(f"Last Modified: {mod_time.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("❌ File not found on Drive.")


✅ Cache found on Drive!
Path: /content/drive/MyDrive/RiverScope/results/width_estimation_new_run/all_experiment_dfs.pkl
Size: 0.31 MB
Last Modified: 2026-05-12 07:09:06
